# 3주차 실습 — rosbag → 라벨링 → 학습

조교 차가 실습실 트랙을 달리며 녹화한 **rosbag**에서 BEV를 만들고, 프레임마다 정답(1 m 앞 점)을 직접 찍어서
2주차와 **같은 형식의 데이터셋**을 만든 뒤 2주차 코드로 학습까지 돌려 봄.

시뮬은 정답이 자동으로 나왔음. 실차는 아무도 정답을 모르니 사람이 찍음. 오늘 할 일은 그 과정 전체.

| 장 | 내용 |
|---|---|
| 0 | 설치 |
| 1 | rosbag 살펴보기 |
| 2 | BEV 만들기 |
| 3 | 라벨링 (클릭) |
| 4 | 데이터셋 (2주차 형식) |
| 5 | 학습 |

라벨이 30장 정도라 성능은 보지 않음. **녹화 → 라벨 → 데이터셋 → 학습이 끝까지 돌아가는 것**이 목표.

## 0. 설치

실행 전에 메뉴에서 두 가지. **런타임 > 런타임 유형 변경 > T4 GPU**, **파일 > 드라이브에 사본 저장**.

첫 셀이 레포를 clone 함. 레포 안에 오늘 쓸 bag(`camreal/sample/week3/`)이 같이 들어 있음 (약 60 MB).

In [ ]:
%%bash
# 코랩 첫 실행이면 clone + 설치, 다시 실행이면 pull 만
set -e
cd /content
[ -d f1tenth_camreal ] || git clone -q --branch main https://github.com/minjai345/f1tenth_camreal.git
cd f1tenth_camreal
git pull -q --ff-only
pip install -q -r camsim/requirements.txt rosbags      # rosbags: ROS 없이 rosbag 을 읽는 파이썬 패키지

In [ ]:
import os, sys, numpy as np, cv2, torch, pandas as pd
import matplotlib.pyplot as plt
REPO = os.environ.get("WEEK3_REPO", "/content/f1tenth_camreal")
os.chdir(REPO)
sys.path[:0] = [REPO]
from camsim import config, dataset, model, train
from camreal import week3_lab as lab

plt.rcParams["axes.unicode_minus"] = False       # 코랩 기본 폰트에 한글 없음. 그래프 글자는 영어로
os.makedirs("out", exist_ok=True)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
cfg = config.load()                              # 2주차 기본 설정 (BEV 300 x 380 px, 1 px = 1 cm)

BAG = lab.SAMPLE / "run_train2_part1_2hz"        # 내 bag 을 쓰려면 이 경로를 바꿈
CAL = lab.SAMPLE / "car.yaml"                    # 그 bag 을 녹화한 차의 바닥 캘리브레이션
print("device:", DEVICE, "| bag:", BAG.name)

## 1. rosbag 살펴보기

rosbag 은 ROS 토픽으로 오간 메시지를 **받은 시각과 함께 그대로** 저장한 것. 폴더 하나가 bag 하나이고,
안에 `metadata.yaml`(목록)과 `.db3`(메시지)가 있음.

조교가 차에서 녹화한 명령 (차 안에서 Ctrl+C 로 끝냄):

```bash
ros2 bag record --storage sqlite3 \
  --qos-profile-overrides-path camreal/config/recording_qos.yaml \
  --output data/bags/run_train2 /flir_camera/image_raw /flir_camera/camera_info
```

녹화는 83초, 1.5 m/s 이하로 천천히. 그중 차선이 잘 보이는 15초만 잘라서 0.5초에 한 장만 남겼고(레포에 넣으려고),
사람 얼굴은 모자이크함. 아래 셀은 차에서 `ros2 bag info` 로 보는 것과 같은 내용을 보여 줌.

In [ ]:
print(sorted(os.listdir(BAG)))
info = lab.bag_summary(BAG)
print(f"길이 {info['duration_s']:.1f} s")
pd.DataFrame(info["topics"])

In [ ]:
# 0.5초마다 한 장. 카메라 원본은 bayer 패턴이라 색을 복원해서 보여 줌
frames = lab.read_frames(BAG, every_s=0.5)
print(len(frames), "장 |", frames[0]["bgr"].shape)
plt.figure(figsize=(10, 6.5)); plt.imshow(frames[0]["bgr"][..., ::-1]); plt.axis("off"); plt.title("camera image")
plt.show()

## 2. BEV 만들기

앞 시간(IPM)에서 본 것처럼 바닥 캘리브레이션(`car.yaml`의 `H_i2g`)으로 카메라 영상을 위에서 본 BEV 로 폄.
차의 주행 노드와 **같은 함수**를 쓰고, 2주차 시뮬 BEV 와 같은 규격(300 x 380 px, 1 px = 1 cm, 앞 0.2~4.0 m)임.
학습 카메라가 못 보는 구석은 회색으로 칠함.

In [ ]:
bevs = lab.make_bevs(frames, CAL, cfg)
fig, axes = plt.subplots(2, 7, figsize=(14, 6.5))
for ax, k in zip(axes.flat, range(0, len(bevs), 2)):
    ax.imshow(bevs[k][..., ::-1]); ax.set_title(str(k)); ax.axis("off")
plt.tight_layout(); plt.show()

## 3. 라벨링

아래 셀을 실행하면 클릭 화면이 뜸. 십자 = 뒷바퀴 축, 점선 원 = 뒷바퀴 축에서 1 m.

1. **원이 좌우 테이프의 가운데와 만나는 곳을 클릭** (점이 원 위에 붙음, 드래그로 옮김)
2. 맞으면 **Enter**(승인) → 다음 프레임. 가운데를 모르겠으면(테이프가 안 보임, 흐림) **X**(제외)
3. **←, →** 로 앞뒤 이동, "다음 미작업" 으로 남은 프레임

키보드가 안 먹으면 화면을 한 번 클릭. 라벨은 누를 때마다 `out/week3_labels.json` 에 저장되고, 셀을 다시 실행해도 남아 있음.

In [ ]:
labeler = lab.Labeler(bevs, cfg, "out/week3_labels.json")
labeler.show()

In [ ]:
labeler.counts()      # 라벨링이 끝나면 실행해서 확인

## 4. 데이터셋 (2주차 형식)

승인한 프레임만 모아서 2주차 2장 "데이터셋 저장"과 **같은 폴더**를 만듦: `images/*.png` + `labels.csv`.

`labels.csv` 의 열은 2주차와 같음(`file, x, y, theta, wp_x, wp_y`). 실차는 차가 맵 어디에 있는지 모르니
`x, y, theta` 는 비워 둠(`nan`). 학습에는 `wp_x, wp_y` (1 m 앞 점)만 씀.

In [ ]:
n = lab.write_dataset(bevs, labeler.labels, "out/week3_real", prefix="run_train2")
print(n, "장 -> out/week3_real")
pd.read_csv("out/week3_real/labels.csv").head()

## 5. 학습

2주차 3장과 **같은 함수**(`DiskDataset`, `train`)에 폴더 경로만 바꿔 넣음. 2주차처럼 20% 는 검증용으로 떼어 둠.

30장으로는 모델이 장면을 외워 버림(train loss 는 내려가도 실력은 안 늚). 실제로 쓰려면 여러 주행·조명에서
수백~수천 장을 모아야 함. 오늘은 흐름 확인까지.

In [ ]:
train_ds = dataset.DiskDataset("out/week3_real", cfg, "train", val_frac=0.2)
val_ds = dataset.DiskDataset("out/week3_real", cfg, "val", val_frac=0.2)
print("train", len(train_ds), "/ val", len(val_ds))
STEPS, BATCH = 300, min(8, len(train_ds))
net, history = train.train(None, cfg, steps=STEPS, batch_size=BATCH, lr=1e-3, device=DEVICE,
                           dataset=train_ds, val_dataset=val_ds, log_every=25)
h = pd.DataFrame(history)
plt.plot(h["step"], h["loss"], label="train"); plt.plot(h["step"], h["val_loss"], label="val")
plt.yscale("log"); plt.xlabel("step"); plt.ylabel("loss"); plt.legend(); plt.show()

In [ ]:
pred = model.Predictor(net, cfg, DEVICE)
res = train.evaluate_dataset(pred, val_ds)
print(f"검증 {res['n']}장 평균 오차 {res['mean_m'] * 100:.1f} cm (최대 {res['max_m'] * 100:.1f} cm)")

b = cfg.bev
pixel = lambda p: (int((b.y_range_m[1] - p[1]) / b.resolution_m), int((b.x_range_m[1] - p[0]) / b.resolution_m))
fig, axes = plt.subplots(1, min(4, len(val_ds)), figsize=(12, 4.5), squeeze=False)
for ax, i in zip(axes[0], range(len(val_ds))):
    img = val_ds.load_image(i).copy()
    cv2.circle(img, pixel(val_ds.wps[val_ds.idx[i]]), 7, (0, 200, 0), -1)       # 초록: 내가 찍은 정답
    cv2.circle(img, pixel(pred.predict(img)), 5, (255, 0, 255), -1)             # 자홍: 모델 예측
    ax.imshow(img[..., ::-1]); ax.axis("off")
plt.suptitle("green: label   magenta: prediction"); plt.show()

## 정리

- 녹화(rosbag) → 프레임 추출 → BEV → 사람이 정답을 찍음 → 2주차 형식 데이터셋 → 2주차 코드로 학습
- 내 bag 으로 하려면: 0장 다음 셀의 `BAG` 을 내 bag 폴더로, `CAL` 을 그 차의 `car.yaml` 로 바꾸고 1장부터 다시 실행.
  원본 bag 은 1분에 약 5 GB 라 Colab 에 올리기 전에 0.5초 간격으로 줄이는 게 좋음 (`camreal/tools/decimate_bag.py`, 차에서 실행)
- 30장이 아니라 3000장이 필요하다면 어떤 장면을 더 녹화해야 할까? (조명, 코너, 반대 방향, 다른 트랙)